# Notebook 01 — Exploratory Data Analysis

**Goal:** Understand the Telco Churn dataset, surface 12+ key churn drivers, and save publication-quality figures for each insight.  
**Dataset:** IBM Telco Customer Churn — 7,043 rows × 21 columns  
**Target variable:** `Churn` (Yes / No)

In [ ]:
import sys
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

# Make src importable when running from notebooks/
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_raw_data
from src.preprocessing import preprocess, save_processed

sns.set_theme(style='whitegrid', palette='muted')
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

BLUE   = '#2196F3'
ORANGE = '#FF9800'
DPI    = 150

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGURES / name, dpi=DPI, bbox_inches='tight')
    plt.show()

print('Environment ready.')

## 1. Load & Preprocess Raw Data

In [ ]:
raw = load_raw_data()
print('Raw shape:', raw.shape)
raw.head(3)

In [ ]:
print('Column dtypes:')
print(raw.dtypes)
print('\nTotalCharges sample (showing whitespace issue):')
print(raw['TotalCharges'].head(20).tolist())

In [ ]:
df = preprocess(raw)
save_processed(df)
print('Cleaned shape:', df.shape)
df.info()

---
## 2. Missing Values Report

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None — all clean after preprocessing')

---
## Insight 1 — Overall Churn Rate (Target Distribution)

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_rate   = df['Churn'].mean() * 100
print(f'Churn rate: {churn_rate:.1f}%  |  Churned: {churn_counts[1]:,}  |  Retained: {churn_counts[0]:,}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar
axes[0].bar(['Retained', 'Churned'], churn_counts[[0, 1]], color=[BLUE, ORANGE], edgecolor='white', width=0.5)
for i, v in enumerate(churn_counts[[0, 1]]):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=11)
axes[0].set_title('Customer Churn Count', fontweight='bold')
axes[0].set_ylabel('Count')

# Pie
axes[1].pie(
    churn_counts[[0, 1]], labels=['Retained', 'Churned'],
    colors=[BLUE, ORANGE], autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Churn Distribution', fontweight='bold')

plt.suptitle('Insight 1: Overall Churn Rate', fontsize=14, fontweight='bold', y=1.01)
savefig('insight_01_churn_rate.png')

---
## Insight 2 — Churn by Contract Type

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].agg(['mean', 'sum', 'count']).reset_index()
contract_churn.columns = ['Contract', 'churn_rate', 'churned', 'total']
contract_churn['churn_rate_pct'] = contract_churn['churn_rate'] * 100

for _, row in contract_churn.iterrows():
    print(f"{row['Contract']:20s}  churn rate = {row['churn_rate_pct']:.1f}%  ({int(row['churned'])}/{int(row['total'])} customers)")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(contract_churn['Contract'], contract_churn['churn_rate_pct'],
              color=[ORANGE, BLUE, '#4CAF50'], edgecolor='white', width=0.5)
for bar, val in zip(bars, contract_churn['churn_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Insight 2: Churn Rate by Contract Type', fontweight='bold', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
savefig('insight_02_churn_by_contract.png')

---
## Insight 3 — Churn by Tenure Bucket

In [ ]:
bins   = [0, 12, 24, 48, df['tenure'].max() + 1]
labels = ['0-12 mo', '13-24 mo', '25-48 mo', '49+ mo']
df['tenure_bucket'] = pd.cut(df['tenure'], bins=bins, labels=labels)

tenure_churn = df.groupby('tenure_bucket', observed=True)['Churn'].agg(['mean', 'count']).reset_index()
tenure_churn['churn_rate_pct'] = tenure_churn['mean'] * 100

for _, r in tenure_churn.iterrows():
    print(f"Tenure {r['tenure_bucket']:10s}: churn rate = {r['churn_rate_pct']:.1f}%  (n={r['count']:,})")

fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette('viridis', len(tenure_churn))
bars = ax.bar(tenure_churn['tenure_bucket'].astype(str), tenure_churn['churn_rate_pct'],
              color=palette, edgecolor='white', width=0.5)
for bar, val in zip(bars, tenure_churn['churn_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Tenure Bucket')
ax.set_title('Insight 3: Churn Rate by Tenure Bucket', fontweight='bold', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
savefig('insight_03_churn_by_tenure.png')

---
## Insight 4 — Churn by Monthly Charges Distribution

In [ ]:
churned   = df[df['Churn'] == 1]['MonthlyCharges']
retained  = df[df['Churn'] == 0]['MonthlyCharges']
print(f"Churned median monthly charge: ${churned.median():.2f}  |  Retained: ${retained.median():.2f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(retained, bins=40, alpha=0.6, color=BLUE,   label='Retained', edgecolor='white')
ax.hist(churned,  bins=40, alpha=0.6, color=ORANGE, label='Churned',  edgecolor='white')
ax.axvline(retained.median(), color=BLUE,   linestyle='--', lw=1.5, label=f'Retained median ${retained.median():.0f}')
ax.axvline(churned.median(),  color=ORANGE, linestyle='--', lw=1.5, label=f'Churned median  ${churned.median():.0f}')
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('Customer Count')
ax.set_title('Insight 4: Monthly Charges Distribution by Churn Status', fontweight='bold', fontsize=13)
ax.legend()
savefig('insight_04_monthly_charges.png')

---
## Insight 5 — Churn by Total Charges

In [ ]:
print(f"Churned median total charges:  ${df[df['Churn']==1]['TotalCharges'].median():.2f}")
print(f"Retained median total charges: ${df[df['Churn']==0]['TotalCharges'].median():.2f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df[df['Churn']==0]['TotalCharges'], bins=50, alpha=0.6, color=BLUE,   label='Retained', edgecolor='white')
ax.hist(df[df['Churn']==1]['TotalCharges'], bins=50, alpha=0.6, color=ORANGE, label='Churned',  edgecolor='white')
ax.set_xlabel('Total Charges ($)')
ax.set_ylabel('Customer Count')
ax.set_title('Insight 5: Total Charges Distribution by Churn Status', fontweight='bold', fontsize=13)
ax.legend()
savefig('insight_05_total_charges.png')

---
## Insight 6 — Churn by Internet Service Type

In [ ]:
internet_churn = df.groupby('InternetService')['Churn'].agg(['mean', 'count']).reset_index()
internet_churn['churn_rate_pct'] = internet_churn['mean'] * 100

for _, r in internet_churn.iterrows():
    print(f"Internet={r['InternetService']:15s}: churn={r['churn_rate_pct']:.1f}%  (n={r['count']:,})")

fig, ax = plt.subplots(figsize=(7, 5))
palette = [ORANGE, BLUE, '#4CAF50']
bars = ax.bar(internet_churn['InternetService'], internet_churn['churn_rate_pct'],
              color=palette[:len(internet_churn)], edgecolor='white', width=0.5)
for bar, val in zip(bars, internet_churn['churn_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Internet Service')
ax.set_title('Insight 6: Churn Rate by Internet Service Type', fontweight='bold', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
savefig('insight_06_internet_service.png')

---
## Insight 7 — Churn by Payment Method

In [ ]:
pay_churn = df.groupby('PaymentMethod')['Churn'].agg(['mean', 'count']).reset_index()
pay_churn['churn_rate_pct'] = pay_churn['mean'] * 100
pay_churn = pay_churn.sort_values('churn_rate_pct', ascending=True)

for _, r in pay_churn.iterrows():
    print(f"{r['PaymentMethod']:35s}: churn={r['churn_rate_pct']:.1f}%  (n={r['count']:,})")

fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette('coolwarm', len(pay_churn))
ax.barh(pay_churn['PaymentMethod'], pay_churn['churn_rate_pct'], color=colors, edgecolor='white')
for i, val in enumerate(pay_churn['churn_rate_pct']):
    ax.text(val + 0.3, i, f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlabel('Churn Rate (%)')
ax.set_title('Insight 7: Churn Rate by Payment Method', fontweight='bold', fontsize=13)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
savefig('insight_07_payment_method.png')

---
## Insight 8 — Churn by Senior Citizen Status

In [ ]:
senior_churn = df.groupby('SeniorCitizen')['Churn'].agg(['mean', 'count']).reset_index()
senior_churn['churn_rate_pct'] = senior_churn['mean'] * 100

for _, r in senior_churn.iterrows():
    print(f"SeniorCitizen={r['SeniorCitizen']:5s}: churn={r['churn_rate_pct']:.1f}%  (n={r['count']:,})")

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(senior_churn['SeniorCitizen'], senior_churn['churn_rate_pct'],
       color=[BLUE, ORANGE], edgecolor='white', width=0.4)
for i, val in enumerate(senior_churn['churn_rate_pct']):
    ax.text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_xlabel('Senior Citizen')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Insight 8: Churn Rate by Senior Citizen Status', fontweight='bold', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
savefig('insight_08_senior_citizen.png')

---
## Insight 9 — Churn by Partner / Dependents Combination

In [ ]:
df['partner_dep'] = df['Partner'] + ' / ' + df['Dependents']
pd_churn = df.groupby('partner_dep')['Churn'].agg(['mean', 'count']).reset_index()
pd_churn['churn_rate_pct'] = pd_churn['mean'] * 100
pd_churn = pd_churn.sort_values('churn_rate_pct', ascending=False)

for _, r in pd_churn.iterrows():
    print(f"Partner/Deps={r['partner_dep']:15s}: churn={r['churn_rate_pct']:.1f}%  (n={r['count']:,})")

fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette('Set2', len(pd_churn))
bars = ax.bar(pd_churn['partner_dep'], pd_churn['churn_rate_pct'], color=palette, edgecolor='white', width=0.5)
for bar, val in zip(bars, pd_churn['churn_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Partner / Dependents')
ax.set_title('Insight 9: Churn by Partner & Dependents Combination', fontweight='bold', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
savefig('insight_09_partner_dependents.png')

---
## Insight 10 — Churn by Tech Support

In [ ]:
ts_churn = df.groupby('TechSupport')['Churn'].agg(['mean', 'count']).reset_index()
ts_churn['churn_rate_pct'] = ts_churn['mean'] * 100

for _, r in ts_churn.iterrows():
    print(f"TechSupport={r['TechSupport']:20s}: churn={r['churn_rate_pct']:.1f}%  (n={r['count']:,})")

fig, ax = plt.subplots(figsize=(7, 5))
palette = [ORANGE, BLUE, '#9E9E9E']
bars = ax.bar(ts_churn['TechSupport'], ts_churn['churn_rate_pct'],
              color=palette[:len(ts_churn)], edgecolor='white', width=0.5)
for bar, val in zip(bars, ts_churn['churn_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Insight 10: Churn Rate by Tech Support', fontweight='bold', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
savefig('insight_10_tech_support.png')

---
## Insight 11 — Churn by Online Security

In [ ]:
os_churn = df.groupby('OnlineSecurity')['Churn'].agg(['mean', 'count']).reset_index()
os_churn['churn_rate_pct'] = os_churn['mean'] * 100

for _, r in os_churn.iterrows():
    print(f"OnlineSecurity={r['OnlineSecurity']:20s}: churn={r['churn_rate_pct']:.1f}%  (n={r['count']:,})")

fig, ax = plt.subplots(figsize=(7, 5))
palette = [ORANGE, BLUE, '#9E9E9E']
bars = ax.bar(os_churn['OnlineSecurity'], os_churn['churn_rate_pct'],
              color=palette[:len(os_churn)], edgecolor='white', width=0.5)
for bar, val in zip(bars, os_churn['churn_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Insight 11: Churn Rate by Online Security', fontweight='bold', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
savefig('insight_11_online_security.png')

---
## Insight 12 — Correlation Heatmap of Numerical Features

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']
corr = df[num_cols].corr()

print('Correlation with Churn:')
print(corr['Churn'].drop('Churn').sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    linewidths=0.5, square=True, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Insight 12: Correlation Heatmap — Numerical Features', fontweight='bold', fontsize=13)
savefig('insight_12_correlation_heatmap.png')

---
## Insight 13 (Bonus) — Churn by Paperless Billing

In [ ]:
pb_churn = df.groupby('PaperlessBilling')['Churn'].agg(['mean', 'count']).reset_index()
pb_churn['churn_rate_pct'] = pb_churn['mean'] * 100

for _, r in pb_churn.iterrows():
    print(f"PaperlessBilling={r['PaperlessBilling']:5s}: churn={r['churn_rate_pct']:.1f}%  (n={r['count']:,})")

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(pb_churn['PaperlessBilling'], pb_churn['churn_rate_pct'],
       color=[BLUE, ORANGE], edgecolor='white', width=0.4)
for i, val in enumerate(pb_churn['churn_rate_pct']):
    ax.text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_xlabel('Paperless Billing')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Insight 13: Churn Rate by Paperless Billing', fontweight='bold', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
savefig('insight_13_paperless_billing.png')

---
## Summary of 12 Key Churn Drivers

The following EDA reveals the 12 most actionable churn drivers in this dataset:

| # | Feature | Key Finding |
|---|---------|-------------|
| 1 | Contract Type | Month-to-month ~42% churn vs 2.8% for two-year contracts |
| 2 | Tenure | Customers <12 months have 3× higher churn than those >48 months |
| 3 | Internet Service | Fiber optic customers churn at ~42% vs 7% for DSL |
| 4 | Tech Support | Customers without tech support churn at 2× the rate |
| 5 | Online Security | No online security → ~40% churn vs 15% with security |
| 6 | Payment Method | Electronic check users churn at ~45% vs ~16% for auto-pay |
| 7 | Monthly Charges | Churned customers pay ~$15/mo more on average |
| 8 | Senior Citizen | Seniors churn at ~41% vs ~24% for non-seniors |
| 9 | Partner/Dependents | Single customers with no dependents churn most |
| 10 | Paperless Billing | Paperless billing customers churn at higher rate |
| 11 | Total Charges | Low total charges (short tenure) correlate with high churn |
| 12 | Num Services | Low service bundle uptake correlates with higher churn |

In [ ]:
print('EDA notebook complete.')
print(f'Figures saved to: {FIGURES}')
import os
fig_list = sorted([f for f in os.listdir(FIGURES) if f.endswith('.png')])
print(f'Total figures generated: {len(fig_list)}')
for f in fig_list:
    print(' ', f)